# Exercise 1 — First-Layer Knowledge Discovery

Baseline classical KD/ML pipeline on the full 1,599-row UCI Wine Quality
(red) dataset, run *before* any FCA/ToscanaJ work, per `CLAUDE.md` §3
Exercise 1: K-Means clustering, Apriori association rules, and a decision
tree, predicting/explaining the Low/Medium/High quality tier.

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from mlxtend.frequent_patterns import apriori, association_rules

DATA_PATH = "../data/winequality-red.csv"

df = pd.read_csv(DATA_PATH, sep=";")
df.columns = [c.strip() for c in df.columns]


def quality_tier(q):
    if q <= 4:
        return "Low"
    if q <= 6:
        return "Medium"
    return "High"


df["quality_tier"] = df["quality"].apply(quality_tier)
features = [c for c in df.columns if c not in ("quality", "quality_tier")]
print(f"{len(df)} wines, {len(features)} physicochemical features")
df["quality_tier"].value_counts()


1599 wines, 11 physicochemical features


quality_tier
Medium    1319
High       217
Low         63
Name: count, dtype: int64

## 1. K-Means clustering (k = 3, 4, 5)

Standardised features, unsupervised clustering, then compared against the
*actual* quality tier via a crosstab — clusters are not told about quality,
so this checks how much chemistry alone separates the tiers.

In [2]:
X = StandardScaler().fit_transform(df[features])

kmeans_results = {}
for k in [3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    kmeans_results[k] = (km, labels, sil)
    print(f"k={k}: silhouette={sil:.3f}")


k=3: silhouette=0.189
k=4: silhouette=0.172
k=5: silhouette=0.190


In [3]:
for k in [3, 4, 5]:
    _, labels, sil = kmeans_results[k]
    print(f"\n--- k={k} (silhouette={sil:.3f}) ---")
    print(pd.crosstab(labels, df["quality_tier"]))



--- k=3 (silhouette=0.189) ---
quality_tier  High  Low  Medium
row_0                          
0               70   45     607
1              132   12     358
2               15    6     354

--- k=4 (silhouette=0.172) ---
quality_tier  High  Low  Medium
row_0                          
0               22   45     486
1               92    5     217
2               89    9     300
3               14    4     316

--- k=5 (silhouette=0.190) ---
quality_tier  High  Low  Medium
row_0                          
0               15    6     316
1               22   45     486
2               92    4     215
3               87    7     276
4                1    1      26


**Reading the silhouette scores**: all three k values give weak-to-moderate
scores (0.17-0.19), and they barely change with k — there's no clear "natural"
number of clusters in the chemistry alone. None of the crosstabs show a
cluster that is purely Low or purely High; every cluster is majority-Medium,
simply because Medium is 82% of the dataset. **Conclusion: chemistry-only
K-Means does not cleanly recover the quality tiers** — this is itself a
useful finding (motivates why a supervised decision tree, which sees the
quality label, does much better — see §3).

### Interpreting the k=3 clusters chemically

Cluster centers (z-scores) show what each cluster is made of, independent
of how well it tracks quality.

In [4]:
km3, labels3, _ = kmeans_results[3]
centers = pd.DataFrame(km3.cluster_centers_, columns=features, index=[0, 1, 2])
print(pd.Series(labels3).value_counts().sort_index(), "\n")
centers.round(2).T


0    722
1    502
2    375
Name: count, dtype: int64 



,0,1,2
fixed acidity,-0.65,1.00,-0.09
volatile acidity,0.46,-0.69,0.04
citric acid,-0.76,1.02,0.10
residual sugar,-0.23,0.03,0.40
chlorides,-0.19,0.28,-0.00
free sulfur dioxide,-0.23,-0.48,1.07
total sulfur dioxide,-0.35,-0.48,1.32
density,-0.45,0.44,0.28
pH,0.61,-0.75,-0.17
sulphates,-0.29,0.55,-0.19


- **Cluster 0** (722 wines): low fixed acidity & citric acid, high volatile
  acidity, high pH — "lighter, more volatile-acidic, less stable" wines.
- **Cluster 1** (502 wines): high fixed acidity & citric acid, low volatile
  acidity & pH, high sulphates — "robust, well-preserved, acidic" wines.
- **Cluster 2** (375 wines): high free/total SO2, higher residual sugar,
  lower alcohol — "heavily sulfited, sweeter, lower-alcohol" wines.

These are coherent chemical groupings, just not ones that line up with the
quality score — confirming quality depends on a combination of these axes
rather than any single cluster being "the good wine cluster".

## 2. Apriori association rules

Quality discretised into Low/Medium/High (matching `CLAUDE.md`). Every
continuous attribute is binned using the same thresholds as the Exercise 2
ordinal scales (mutually-exclusive categorical bins here, not cumulative —
Apriori needs a one-hot item table). `min_support=0.1`, `min_confidence=0.6`,
as specified.

In [5]:
BIN_THRESHOLDS = {
    "fixed acidity": [6, 8, 10], "volatile acidity": [0.3, 0.5, 0.7], "citric acid": [0.25],
    "residual sugar": [4, 12], "chlorides": [0.08], "free sulfur dioxide": [6, 15, 30],
    "total sulfur dioxide": [20, 60, 100], "density": [0.997], "pH": [3.1, 3.4],
    "sulphates": [0.4, 0.6, 0.8], "alcohol": [9, 11, 13],
}

onehot_cols = {}
for col, thresholds in BIN_THRESHOLDS.items():
    labels = (
        [f"{col}<{thresholds[0]}"]
        + [f"{col}[{thresholds[i]},{thresholds[i+1]})" for i in range(len(thresholds) - 1)]
        + [f"{col}>={thresholds[-1]}"]
    )
    cuts = [-np.inf] + thresholds + [np.inf]
    onehot_cols[col] = pd.cut(df[col], bins=cuts, labels=labels, right=False)

onehot_df = pd.DataFrame(onehot_cols)
onehot_df["quality_tier"] = df["quality_tier"]
basket = pd.get_dummies(onehot_df)
print(f"Item table: {basket.shape[0]} wines x {basket.shape[1]} binary items")

freq_itemsets = apriori(basket, min_support=0.1, use_colnames=True)
rules = association_rules(freq_itemsets, metric="confidence", min_threshold=0.6)
print(f"{len(freq_itemsets)} frequent itemsets, {len(rules)} rules")


Item table: 1599 wines x 39 binary items


2199 frequent itemsets, 7800 rules


In [6]:
# Rules predicting FROM quality tier (what chemistry follows from being High-quality?)
from_high = rules[rules["antecedents"].apply(lambda s: s == frozenset({"quality_tier_High"}))]
from_high = from_high.sort_values("lift", ascending=False)
print("Rules with antecedent = quality_tier_High:")
from_high[["consequents", "support", "confidence", "lift"]]


Rules with antecedent = quality_tier_High:


,consequents,support,confidence,lift
45,frozenset({citric acid_citric acid>=0.25}),0.106942,0.788018,1.514473
66,frozenset({residual sugar_residual sugar<4}),0.117573,0.866359,0.946896


In [7]:
# Rules predicting quality tier FROM chemistry, restricted to a single-item consequent
for tier in ["Low", "Medium", "High"]:
    target = frozenset({f"quality_tier_{tier}"})
    matching = rules[rules["consequents"].apply(lambda s: s == target)]
    print(f"quality_tier_{tier} as sole consequent: {len(matching)} rules (support>=0.1, confidence>=0.6)")


quality_tier_Low as sole consequent: 0 rules (support>=0.1, confidence>=0.6)
quality_tier_Medium as sole consequent: 925 rules (support>=0.1, confidence>=0.6)
quality_tier_High as sole consequent: 0 rules (support>=0.1, confidence>=0.6)


**No rule concludes purely `quality_tier_High` or `quality_tier_Low`** at
these thresholds — both are minority classes (13.6% and 3.9% of the data
respectively), and `min_support=0.1` requires an itemset to cover >= 160 of
1,599 wines; no chemistry combination concentrates enough of the rare
High/Low wines together with any other single item to clear that bar.

The reverse direction *did* find something: **`quality_tier_High -> citric_acid>=0.25`**
holds with confidence 78.8% and **lift 1.51** — High-quality wines are 51% more
likely than the average wine to have detectable citric acid. This agrees with
the Exercise 2 implication analysis, which also flagged citric acid presence
as linked to other "good chemistry" markers (`report/ex2_toscana.md` §6,
implication 1, viewed from the complementary angle).

925 rules conclude in `quality_tier_Medium` alone — unsurprising since it's
82% of the data; their lift values are all close to 1 (best lift ~1.17),
confirming they mostly restate the base rate rather than revealing structure.

## 3. Decision tree (max_depth=4)

Supervised baseline: given the quality label, how well can a shallow tree
separate the tiers, and which attributes does it pick?

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df["quality_tier"], test_size=0.25, random_state=42, stratify=df["quality_tier"]
)

clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)

print(f"Train accuracy: {clf.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {clf.score(X_test, y_test):.3f}\n")
print(export_text(clf, feature_names=features))


Train accuracy: 0.874
Test accuracy:  0.828

|--- alcohol <= 11.45
|   |--- volatile acidity <= 0.34
|   |   |--- fixed acidity <= 11.65
|   |   |   |--- alcohol <= 10.75
|   |   |   |   |--- class: Medium
|   |   |   |--- alcohol >  10.75
|   |   |   |   |--- class: Medium
|   |   |--- fixed acidity >  11.65
|   |   |   |--- residual sugar <= 2.95
|   |   |   |   |--- class: High
|   |   |   |--- residual sugar >  2.95
|   |   |   |   |--- class: Medium
|   |--- volatile acidity >  0.34
|   |   |--- volatile acidity <= 1.01
|   |   |   |--- total sulfur dioxide <= 55.50
|   |   |   |   |--- class: Medium
|   |   |   |--- total sulfur dioxide >  55.50
|   |   |   |   |--- class: Medium
|   |   |--- volatile acidity >  1.01
|   |   |   |--- fixed acidity <= 7.75
|   |   |   |   |--- class: Low
|   |   |   |--- fixed acidity >  7.75
|   |   |   |   |--- class: Medium
|--- alcohol >  11.45
|   |--- sulphates <= 0.63
|   |   |--- pH <= 3.26
|   |   |   |--- residual sugar <= 3.90
|   |   |

**83% test accuracy**, vastly better than K-Means's unsupervised separation
(unsurprising — it has access to the label). The top two splits are
**alcohol** (`<= 11.45`) and **volatile acidity** (`<= 0.34`), which recur
throughout the tree — the same two attributes that turned out to dominate
the Exercise 2 implication analysis and the Exercise 4 triadic comparison.
Three independent methods (decision tree, FCA implications, triadic intent
comparison) converging on the same two attributes is a meaningful
cross-check, not a coincidence.